# CAM Intersection Defence — Multilingual Attack (2-mod & 4-mod)

Defends against a **multilingual typographic attack** (EN text at box-0, ZH text at box-1)
using GradCAM intersection masking.

Two defence variants:
- **2-mod**: intersect GradCAM(EN model, EN text) ∩ GradCAM(ZH model, ZH text)
- **4-mod**: intersect all 4 cross-language combos (EN-EN ∩ EN-ZH ∩ ZH-EN ∩ ZH-ZH)

Results saved to `results/cam_2mod/` and `results/cam_4mod/`.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets',
                'matplotlib', 'Pillow'], check=False)

CompletedProcess(args=['d:\\ian\\2026summer\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow'], returncode=0)

In [2]:
import importlib, sys, os, platform, random, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

LANGS = ['en', 'zh']
CLASSES = {
    'en': ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck'],
    'zh': ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}

RESULTS_DIR_2MOD = 'results/cam_2mod'
RESULTS_DIR_4MOD = 'results/cam_4mod'
CACHE_DIR_4MOD   = os.path.join(RESULTS_DIR_4MOD, 'cache')
CACHE_DIR_2MOD   = os.path.join(RESULTS_DIR_2MOD, 'cache')
for d in [RESULTS_DIR_2MOD, RESULTS_DIR_4MOD, CACHE_DIR_4MOD, CACHE_DIR_2MOD]:
    os.makedirs(d, exist_ok=True)

SETUP_LABEL  = 'multilingual'
ATTACK_LABEL = 'multilingual'

Device: cuda


## 1. Model definitions

In [3]:
def classify(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

def classify_conf(model, imgs, words, batch_size=128):
    """Returns (predictions, max-cosine-sim confidences)."""
    preds, confs = [], []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        sims = imf @ tf.t()
        preds.append(sims.argmax(-1).cpu().numpy())
        confs.append(sims.max(-1).values.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)

def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True, return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'], attention_mask=t['attention_mask'],
                                        token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

MODEL_CLS = {'en': EnCLIP, 'zh': ZhCLIP}
print('Model classes defined:', list(MODEL_CLS.keys()))

Model classes defined: ['en', 'zh']


## 2. Attack helpers

In [4]:
DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
_FONT_CACHE  = {}

def _font_paths():
    if platform.system() == 'Windows':
        winfonts = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk   = os.path.join(winfonts, 'msyh.ttc')
        latin = os.path.join(winfonts, 'arial.ttf')
        if not os.path.exists(latin): latin = cjk
        return (cjk if os.path.exists(cjk) else None,
                latin if os.path.exists(latin) else None)
    for d in ['/usr/share/fonts', '/Library/Fonts', os.path.expanduser('~/.fonts')]:
        for f in ['NotoSansCJK-Regular.ttc', 'NotoSans-Regular.ttf']:
            p = os.path.join(d, f)
            if os.path.exists(p): return p, p
    return None, None

_CJK_FONT, _LAT_FONT = _font_paths()

def _font_for(word):
    return _CJK_FONT if any(ord(c) > 127 for c in word) else _LAT_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:    _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except: _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _rects_overlap(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

def _random_nonoverlapping_rect(rng, box_w, box_h, placed):
    x_hi = max(0, DISPLAY_SIZE - box_w)
    y_hi = max(0, DISPLAY_SIZE - box_h)
    rect_x, rect_y = 0, 0
    for _ in range(64):
        rect_x = rng.randint(0, x_hi) if x_hi > 0 else 0
        rect_y = rng.randint(0, y_hi) if y_hi > 0 else 0
        rect = (rect_x, rect_y, rect_x + box_w, rect_y + box_h)
        if all(not _rects_overlap(rect, p) for p in placed): return rect
    return (rect_x, rect_y, rect_x + box_w, rect_y + box_h)

def _draw_text_box(draw, word, rect, font):
    rx, ry, rx2, ry2 = rect
    bb = draw.textbbox((0, 0), word, font=font)
    draw.rectangle([rx, ry, rx2, ry2], fill='white')
    draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)

print('Font paths:', _CJK_FONT, _LAT_FONT)

Font paths: C:\WINDOWS\Fonts\msyh.ttc C:\WINDOWS\Fonts\arial.ttf


In [5]:
def draw_multilingual_attack(img, en_word, zh_word, img_idx, already_224=False):
    """Place English text at box-0 position, Chinese text at box-1 position."""
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    placed = []
    for box_i, (word, fp) in enumerate([(en_word, _LAT_FONT), (zh_word, _CJK_FONT)]):
        font = _get_font(fp)
        bb   = draw.textbbox((0, 0), word, font=font)
        bw   = (bb[2] - bb[0]) + 2 * PAD
        bh   = (bb[3] - bb[1]) + PAD + 12
        rng  = random.Random(int(img_idx) * NUM_BOXES + box_i)
        rect = _random_nonoverlapping_rect(rng, bw, bh, placed)
        placed.append(rect)
        _draw_text_box(draw, word, rect, font)
    return img

def build_multilingual_attacked_images(base_imgs, img_indices, n_workers=None):
    """Two boxes per image: EN attack word at box-0, ZH attack word at box-1."""
    n_workers = n_workers or min(8, os.cpu_count() or 4)
    tasks = [(im, int(k)) for im, k in zip(base_imgs, img_indices)]
    def _one(args):
        im, img_idx = args
        return draw_multilingual_attack(im,
                                        CLASSES['en'][target[img_idx]],
                                        CLASSES['zh'][target[img_idx]],
                                        img_idx, already_224=True)
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        return list(pool.map(_one, tasks))

print(f'Multilingual attack helper ready: {NUM_BOXES} boxes @ size {FONT_SIZE} (box-0=EN, box-1=ZH)')

Multilingual attack helper ready: 2 boxes @ size 24 (box-0=EN, box-1=ZH)


## 3. Dataset

In [6]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000
assert np.array_equal(true, np.array(_saved['true']))
assert all((true == c).sum() == 100 for c in range(10))

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean     = [im.convert('RGB') for im in rows[image_key]]
print('Upscaling clean images to 224px...', end=' ', flush=True)
t0 = time.time()
clean_224 = [im.resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC) for im in clean]
print(f'{time.time()-t0:.1f}s')

all_idx  = np.arange(len(clean))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean)} images; tune subset = {len(tune_idx)}')

Upscaling clean images to 224px... 0.3s
Loaded 1000 images; tune subset = 100


## 4. Load models + build multilingual attack

In [7]:
models = {}
for lang, cls in MODEL_CLS.items():
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')

# Standard text embeddings: each model with its own language
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in LANGS}

clean_preds = {lang: classify(models[lang], clean_224, CLASSES[lang]) for lang in LANGS}
clean_acc   = {lang: float((clean_preds[lang] == true).mean()) for lang in LANGS}
print('Clean acc:', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})

Loading en... 

d:\ian\2026summer\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


1.3s
Loading zh... 

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

1.5s
Clean acc: {'en': '85.9%', 'zh': '91.4%'}


In [8]:
print('Building multilingual attacked images...')
t0 = time.time()
attacked_imgs = build_multilingual_attacked_images(clean_224, all_idx)
print(f'Done in {time.time()-t0:.1f}s')

preds_attacked_2d = {}   # preds_attacked_2d[ml] for EN/ZH model on multilingual attack
for ml in LANGS:
    preds_attacked_2d[ml] = classify(models[ml], attacked_imgs, CLASSES[ml])
baseline_acc = {ml: float((preds_attacked_2d[ml] == true).mean()) for ml in LANGS}
baseline_asr = {ml: float((preds_attacked_2d[ml] == target).mean()) for ml in LANGS}
print('Baseline acc:', {k: f'{100*v:.1f}%' for k, v in baseline_acc.items()})
print('Baseline ASR:', {k: f'{100*v:.1f}%' for k, v in baseline_asr.items()})

Building multilingual attacked images...
Done in 0.2s
Baseline acc: {'en': '4.3%', 'zh': '7.3%'}
Baseline ASR: {'en': '95.5%', 'zh': '92.7%'}


## 5. GradCAM + masking helpers

EN ViT-B/32 and ZH ViT-B/16 produce CAMs at different patch resolutions (~7×7 vs ~14×14).
Both are resized to 224×224 before intersection.

In [9]:
def _norm_cam(cam):
    cam = cam.relu() if isinstance(cam, torch.Tensor) else np.maximum(cam, 0)
    cam = cam.detach().cpu().numpy() if isinstance(cam, torch.Tensor) else cam
    cam = cam - cam.min()
    mx  = cam.max()
    return cam / mx if mx > 0 else cam

def _cam_from_conv(act, grad):
    w = grad.mean(dim=(2, 3), keepdim=True)
    return _norm_cam((w * act).sum(dim=1).squeeze(0))

def gradcam_en(pil_img, target_idx):
    wrapper = models['en']; acts = {}
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle = wrapper.m.visual.conv1.register_forward_hook(hook)
    x        = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    feat     = wrapper.m.visual(x)
    img_feat = F.normalize(feat, dim=-1)
    score    = (img_feat @ TEXT_EMB['en'][target_idx:target_idx+1].T).squeeze()
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam

def gradcam_zh(pil_img, target_idx):
    wrapper = models['zh']; acts = {}
    patch  = wrapper.m.vision_model.embeddings.patch_embedding
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle = patch.register_forward_hook(hook)
    pv     = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    out    = wrapper.m.get_image_features(pixel_values=pv)
    img_feat = F.normalize(_clip_feat(out), dim=-1)
    score  = (img_feat @ TEXT_EMB['zh'][target_idx:target_idx+1].T).squeeze()
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam

GRADCAM_FN = {'en': gradcam_en, 'zh': gradcam_zh}
print('Standard GradCAM helpers ready.')

Standard GradCAM helpers ready.


In [10]:
def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ) / 255.0

def n_cam_intersection(*cams):
    """Elementwise min of N CAMs after resizing to DISPLAY_SIZE."""
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2,:-2]|pad[:-2,1:-1]|pad[:-2,2:]|
             pad[1:-1,:-2]|pad[1:-1,1:-1]|pad[1:-1,2:]|
             pad[2:,:-2]  |pad[2:,1:-1]  |pad[2:,2:])
    return m

def cam_to_mask(saliency, threshold=0.85, dilate=3):
    thr  = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0: mask = dilate_mask(mask, iterations=dilate)
    return mask

def apply_mask(pil_img, mask, fill='mean_nonmask'):
    pil_img = pil_img.convert('RGB')
    arr = np.array(pil_img)
    h, w = arr.shape[:2]
    if mask.shape != (h, w):
        mask = np.array(
            Image.fromarray(mask.astype(np.uint8) * 255).resize((w, h), Image.NEAREST)
        ) > 127
    out = arr.copy()
    m   = mask.astype(bool)
    if fill == 'mean_nonmask':
        bg = ~m
        mean_color = arr[bg].mean(0) if bg.any() else arr.reshape(-1, 3).mean(0)
        out[m] = mean_color
    return Image.fromarray(out.astype(np.uint8))

def overlay_cam(pil_img, cam, alpha=0.50):
    cam_img = Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (DISPLAY_SIZE, DISPLAY_SIZE), Image.BILINEAR)
    heat  = cm.jet(np.array(cam_img) / 255.0)[:, :, :3]
    base  = np.array(pil_img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE))).astype(np.float32) / 255.0
    blend = np.clip((1 - alpha) * base + alpha * heat, 0, 1)
    return Image.fromarray((blend * 255).astype(np.uint8))

def mask_overlay(pil_img, mask, alpha=0.45):
    arr = np.array(pil_img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE))).astype(np.float32)
    red = np.zeros_like(arr); red[:, :, 0] = 255
    m   = mask.astype(np.float32)[..., None]
    return Image.fromarray((arr * (1 - alpha * m) + red * (alpha * m)).astype(np.uint8))

print('CAM masking helpers ready.')

CAM masking helpers ready.


## 6. Cross-language text embeddings (for 4-mod)

In [11]:
# Cross-language text embeddings:
#   TEXT_EMBS[('en','zh')] = EN model's text encoder applied to ZH class names
#   TEXT_EMBS[('zh','en')] = ZH model's text encoder applied to EN class names
# These are "cross-lingual probes" - each model sees the other language's vocabulary.

with torch.no_grad():
    # EN model encoding ZH class names (EN tokenizer processes CJK glyphs as UNK/subwords)
    _t_zh_via_en = models['en'].tok(
        [TMPL['en'].format(w) for w in CLASSES['zh']]
    ).to(DEVICE)
    TEXT_EMB_EN_ZH = F.normalize(models['en'].m.encode_text(_t_zh_via_en), dim=-1).detach()

    # ZH model encoding EN class names
    _t_en_via_zh = models['zh'].p(
        text=[TMPL['zh'].format(w) for w in CLASSES['en']],
        padding=True, return_tensors='pt',
    ).to(DEVICE)
    _out_en_via_zh = models['zh'].m.get_text_features(
        input_ids=_t_en_via_zh['input_ids'],
        attention_mask=_t_en_via_zh['attention_mask'],
        token_type_ids=_t_en_via_zh.get('token_type_ids'),
    )
    TEXT_EMB_ZH_EN = F.normalize(_clip_feat(_out_en_via_zh), dim=-1).detach()

# Unified lookup: (model_lang, text_lang) -> text_embedding
TEXT_EMBS = {
    ('en', 'en'): TEXT_EMB['en'],     # standard EN
    ('en', 'zh'): TEXT_EMB_EN_ZH,    # EN model, ZH class names
    ('zh', 'en'): TEXT_EMB_ZH_EN,    # ZH model, EN class names
    ('zh', 'zh'): TEXT_EMB['zh'],     # standard ZH
}
print('Cross-language text embeddings computed.')

Cross-language text embeddings computed.


In [12]:
def gradcam_en_with_emb(pil_img, text_emb, target_idx=None):
    """GradCAM using EN model, scored against text_emb[target_idx]."""
    wrapper = models['en']; acts = {}
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle   = wrapper.m.visual.conv1.register_forward_hook(hook)
    x        = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    feat     = wrapper.m.visual(x)
    img_feat = F.normalize(feat, dim=-1)
    sims     = (img_feat @ text_emb.T).squeeze(0)
    if target_idx is None:
        target_idx = int(sims.detach().argmax().item())
    score = sims[target_idx]
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam, target_idx

def gradcam_zh_with_emb(pil_img, text_emb, target_idx=None):
    """GradCAM using ZH model, scored against text_emb[target_idx]."""
    wrapper = models['zh']; acts = {}
    patch   = wrapper.m.vision_model.embeddings.patch_embedding
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle   = patch.register_forward_hook(hook)
    pv       = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    out      = wrapper.m.get_image_features(pixel_values=pv)
    img_feat = F.normalize(_clip_feat(out), dim=-1)
    sims     = (img_feat @ text_emb.T).squeeze(0)
    if target_idx is None:
        target_idx = int(sims.detach().argmax().item())
    score = sims[target_idx]
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam, target_idx

def get_n_cams(pil_img, combos):
    """Compute CAMs for all (model_lang, text_lang) combos. Returns list of cams."""
    cams = []
    for ml, tl in combos:
        emb = TEXT_EMBS[(ml, tl)]
        fn  = gradcam_en_with_emb if ml == 'en' else gradcam_zh_with_emb
        cam, _ = fn(pil_img, emb)
        cams.append(cam)
    return cams

print('Generalised GradCAM helpers ready.')

Generalised GradCAM helpers ready.


## 7. 4-CAM cache

In [13]:
COMBOS_2MOD = [('en','en'), ('zh','zh')]
COMBOS_4MOD = [('en','en'), ('en','zh'), ('zh','en'), ('zh','zh')]

def _cache_path_4mod(condition, n):
    return os.path.join(CACHE_DIR_4MOD, f'cams4_{condition}_n{n}.npz')

def compute_and_cache_cams(condition, indices, combos=COMBOS_4MOD):
    n     = len(indices)
    nmod  = len(combos)
    label = '_'.join(f'{m}{t}' for m, t in combos)
    cfile = os.path.join(CACHE_DIR_4MOD, f'cams_{label}_{condition}_n{n}.npz')
    keys  = [f'cam_{m}_{t}' for m, t in combos]

    if os.path.exists(cfile):
        data = np.load(cfile, allow_pickle=True)
        print(f'Loaded cache {os.path.basename(cfile)}')
        return {k: data[k] for k in keys}, np.array(data['indices'])

    # Build attacked images for this condition
    if condition == 'multi_attack':
        imgs = [attacked_imgs[i] for i in indices]
    elif condition == 'uni_attack':
        imgs = [attacked_imgs[i] for i in indices]
    elif condition == 'clean':
        imgs = [clean_224[i] for i in indices]
    else:
        raise ValueError(f'Unknown condition: {condition}')

    cam_lists = {k: [] for k in keys}
    t0 = time.time()
    for j, img in enumerate(imgs):
        for (ml, tl), k in zip(combos, keys):
            emb = TEXT_EMBS[(ml, tl)]
            fn  = gradcam_en_with_emb if ml == 'en' else gradcam_zh_with_emb
            cam, _ = fn(img, emb)
            cam_lists[k].append(cam)
        if (j + 1) % 50 == 0:
            print(f'  CAM {j+1}/{n} [{time.time()-t0:.1f}s]')

    data_to_save = {k: np.stack(cam_lists[k]) for k in keys}
    data_to_save['indices'] = np.array(indices)
    np.savez(cfile, **data_to_save)
    print(f'Saved cache {os.path.basename(cfile)} [{time.time()-t0:.1f}s]')
    return {k: data_to_save[k] for k in keys}, np.array(indices)

print('4-CAM cache helpers ready.')

4-CAM cache helpers ready.


## 8. Threshold sweep on 100-image tune subset

In [14]:
# Alias for the cache to work with both 2-mod and 4-mod
cam_2mod_cache = {}
cam_4mod_cache = {}
for cond, label in [('multi_attack', 'multilingual attack'), ('clean', 'clean')]:
    imgs_cond = attacked_imgs if cond == 'multi_attack' else clean_224
    print(f'Computing 4-mod CAMs for {label} (tune subset)...')
    cd4, _ = compute_and_cache_cams(cond, tune_idx, combos=COMBOS_4MOD)
    cam_4mod_cache[cond] = cd4
    # 2-mod uses only the 'en_en' and 'zh_zh' cams
    cam_2mod_cache[cond] = {k: cd4[k] for k in ['cam_en_en', 'cam_zh_zh']}

Computing 4-mod CAMs for multilingual attack (tune subset)...
  CAM 50/100 [3.7s]
  CAM 100/100 [7.0s]
Saved cache cams_enen_enzh_zhen_zhzh_multi_attack_n100.npz [7.0s]
Computing 4-mod CAMs for clean (tune subset)...
  CAM 50/100 [3.4s]
  CAM 100/100 [6.7s]
Saved cache cams_enen_enzh_zhen_zhzh_clean_n100.npz [6.7s]


In [15]:
THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]

def sweep_one(cam_data, combos, imgs_tune, label):
    rows = []
    base = {ml: float((preds_attacked_2d[ml][tune_idx] == true[tune_idx]).mean()) for ml in LANGS}
    for thr in THRESHOLDS:
        sal_list = []
        for j in range(len(tune_idx)):
            cams_j = [cam_data[f'cam_{m}_{t}'][j] for m, t in combos]
            sal_list.append(n_cam_intersection(*cams_j))
        masks  = [cam_to_mask(s, thr, dilate=3) for s in sal_list]
        masked = [apply_mask(imgs_tune[j], masks[j]) for j in range(len(tune_idx))]
        for ml in LANGS:
            p = classify(models[ml], masked, CLASSES[ml])
            t_sub = true[tune_idx]; tgt_sub = target[tune_idx]
            rows.append({
                'variant': label, 'model': ml, 'threshold': thr,
                'baseline_acc': base[ml],
                'masked_acc':   float((p == t_sub).mean()),
                'masked_asr':   float((p == tgt_sub).mean()),
                'coverage':     float(np.mean([m.mean() for m in masks])),
            })
    best = max([r for r in rows if r['model'] == 'en'], key=lambda r: r['masked_acc'])
    print(f'[{label}] best thr={best["threshold"]}  '
          f'acc {100*best["baseline_acc"]:.1f}% -> {100*best["masked_acc"]:.1f}%')
    return rows, best['threshold']

imgs_tune = [attacked_imgs[i] for i in tune_idx]
rows_2mod, BEST_THR_2MOD = sweep_one(cam_2mod_cache['multi_attack'], COMBOS_2MOD, imgs_tune, '2-mod')
rows_4mod, BEST_THR_4MOD = sweep_one(cam_4mod_cache['multi_attack'], COMBOS_4MOD, imgs_tune, '4-mod')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (rows, label) in zip(axes, [(rows_2mod, '2-mod'), (rows_4mod, '4-mod')]):
    for ml, color in zip(LANGS, ['C0', 'C1']):
        xs  = THRESHOLDS
        ys  = [r['masked_acc'] for r in rows if r['model'] == ml]
        asr = [r['masked_asr'] for r in rows if r['model'] == ml]
        ax.plot(xs, [100*y for y in ys],  'o-', color=color, label=f'{ml} acc')
        ax.plot(xs, [100*y for y in asr], 's--', color=color, alpha=0.6, label=f'{ml} ASR')
    ax.set_title(f'{label} threshold sweep'); ax.set_xlabel('Percentile threshold')
    ax.set_ylabel('%'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Threshold sweep — multilingual attack (100-image tune subset)', fontsize=11)
plt.tight_layout()
plt.savefig('results/cam_2mod/threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved -> threshold_sweep.png')

[2-mod] best thr=0.85  acc 6.0% -> 30.0%
[4-mod] best thr=0.9  acc 6.0% -> 32.0%
Saved -> threshold_sweep.png


## 9. Full evaluation on 1000 images

In [16]:
print('Computing full 4-mod CAM cache (1000 images)...')
cd4_full, _ = compute_and_cache_cams('multi_attack', all_idx, combos=COMBOS_4MOD)
cd4_clean_full, _ = compute_and_cache_cams('clean', all_idx, combos=COMBOS_4MOD)
cd2_full  = {k: cd4_full[k]  for k in ['cam_en_en', 'cam_zh_zh']}
cd2_clean = {k: cd4_clean_full[k] for k in ['cam_en_en', 'cam_zh_zh']}

Computing full 4-mod CAM cache (1000 images)...
  CAM 50/1000 [3.4s]
  CAM 100/1000 [6.8s]
  CAM 150/1000 [10.1s]
  CAM 200/1000 [13.6s]
  CAM 250/1000 [16.9s]
  CAM 300/1000 [20.3s]
  CAM 350/1000 [23.8s]
  CAM 400/1000 [27.2s]
  CAM 450/1000 [30.7s]
  CAM 500/1000 [34.1s]
  CAM 550/1000 [37.5s]
  CAM 600/1000 [40.9s]
  CAM 650/1000 [44.3s]
  CAM 700/1000 [47.6s]
  CAM 750/1000 [51.0s]
  CAM 800/1000 [54.4s]
  CAM 850/1000 [57.7s]
  CAM 900/1000 [61.0s]
  CAM 950/1000 [64.4s]
  CAM 1000/1000 [67.8s]
Saved cache cams_enen_enzh_zhen_zhzh_multi_attack_n1000.npz [67.8s]
  CAM 50/1000 [3.3s]
  CAM 100/1000 [6.7s]
  CAM 150/1000 [10.1s]
  CAM 200/1000 [13.6s]
  CAM 250/1000 [17.0s]
  CAM 300/1000 [20.4s]
  CAM 350/1000 [23.9s]
  CAM 400/1000 [27.4s]
  CAM 450/1000 [31.0s]
  CAM 500/1000 [34.5s]
  CAM 550/1000 [37.9s]
  CAM 600/1000 [41.3s]
  CAM 650/1000 [44.6s]
  CAM 700/1000 [47.6s]
  CAM 750/1000 [50.9s]
  CAM 800/1000 [54.0s]
  CAM 850/1000 [57.2s]
  CAM 900/1000 [60.3s]
  CAM 950/1000 

In [17]:
def run_cam_eval(cam_data, cam_data_clean, combos, best_thr, n_mods, results_dir):
    """Full CAM defence evaluation."""
    sal_list = [n_cam_intersection(*[cam_data[f'cam_{m}_{t}'][j] for m, t in combos])
                for j in range(len(all_idx))]
    masks        = [cam_to_mask(s, best_thr, dilate=3) for s in sal_list]
    masked_imgs  = [apply_mask(attacked_imgs[j], masks[j]) for j in range(len(all_idx))]

    defense_res  = {}
    for ml in LANGS:
        base_p = preds_attacked_2d[ml]
        new_p  = classify(models[ml], masked_imgs, CLASSES[ml])
        wrong  = base_p != true
        defense_res[ml] = {
            'acc':           float((new_p == true).mean()),
            'asr':           float((new_p == target).mean()),
            'recovery_rate': float((wrong & (new_p == true)).sum() / wrong.sum()) if wrong.any() else 0.0,
            'baseline_acc':  float((base_p == true).mean()),
            'baseline_asr':  float((base_p == target).mean()),
        }

    # Clean degradation
    clean_sal    = [n_cam_intersection(*[cam_data_clean[f'cam_{m}_{t}'][j] for m, t in combos])
                    for j in range(len(all_idx))]
    clean_masks  = [cam_to_mask(s, best_thr, dilate=3) for s in clean_sal]
    clean_masked = [apply_mask(clean_224[j], clean_masks[j]) for j in range(len(all_idx))]
    clean_deg    = {}
    for ml in LANGS:
        cp = classify(models[ml], clean_masked, CLASSES[ml])
        clean_deg[ml] = {
            'baseline_acc': clean_acc[ml],
            'masked_acc':   float((cp == true).mean()),
            'delta_acc':    float((cp == true).mean()) - clean_acc[ml],
        }

    coverage = float(np.mean([m.mean() for m in masks]))
    results  = {
        'setup':             'multilingual',
        'method':            f'cam_{n_mods}mod',
        'attack':            'multilingual',
        'n_images':          len(all_idx),
        'best_threshold':    best_thr,
        'combos':            str(combos),
        'inference_cost':    2 + 2 * n_mods,
        'clean_acc':         clean_acc,
        'baseline_acc':      {ml: defense_res[ml]['baseline_acc'] for ml in LANGS},
        'baseline_asr':      {ml: defense_res[ml]['baseline_asr'] for ml in LANGS},
        'defense':           defense_res,
        'clean_degradation': clean_deg,
        'coverage_mean':     coverage,
        'defense_acc_mean':  float(np.mean([defense_res[ml]['acc']  for ml in LANGS])),
        'defense_asr_mean':  float(np.mean([defense_res[ml]['asr']  for ml in LANGS])),
    }
    out_path = f'{results_dir}/confusion_results_cam_defense.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f'Saved -> {out_path}')
    return results, masked_imgs, masks

print('Running 2-mod full eval...')
res_2mod, masked_2mod, masks_2mod = run_cam_eval(
    cd2_full, cd2_clean, COMBOS_2MOD, BEST_THR_2MOD, 2, RESULTS_DIR_2MOD)
print('Running 4-mod full eval...')
res_4mod, masked_4mod, masks_4mod = run_cam_eval(
    cd4_full, cd4_clean_full, COMBOS_4MOD, BEST_THR_4MOD, 4, RESULTS_DIR_4MOD)

Running 2-mod full eval...
Saved -> results/cam_2mod/confusion_results_cam_defense.json
Running 4-mod full eval...
Saved -> results/cam_4mod/confusion_results_cam_defense.json


## 10. Visual diagnostics

In [18]:
def plot_delta(results, label, save_path):
    fig, ax = plt.subplots(figsize=(4, 3))
    vals = np.array([[results['defense'][ml]['acc'] - results['defense'][ml]['baseline_acc']]
                      for ml in LANGS]) * 100
    im = ax.imshow(vals, vmin=-20, vmax=60, cmap='RdYlGn')
    ax.set_xticks([0]); ax.set_xticklabels(['multilingual attack'])
    ax.set_yticks(range(len(LANGS))); ax.set_yticklabels([f'model_{l}' for l in LANGS])
    ax.set_title(f'Accuracy delta after {label} (pp)')
    for i, ml in enumerate(LANGS):
        ax.text(0, i, f'{vals[i,0]:+.1f}', ha='center', va='center', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, format=lambda x, _: f'{x:+.0f}pp')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved -> {save_path}')

plot_delta(res_2mod, '2-mod CAM masking', f'{RESULTS_DIR_2MOD}/accuracy_delta_matrix.png')
plot_delta(res_4mod, '4-mod CAM masking', f'{RESULTS_DIR_4MOD}/accuracy_delta_matrix.png')

# Mask examples: 5 images x 7 columns for 2-mod
cam_select = [(c, int(np.where(true == c)[0][0])) for c in range(5)]
fig, axes  = plt.subplots(5, 6, figsize=(14, 18))
col_titles = ['Attacked', 'EN-EN CAM', 'ZH-ZH CAM', 'Intersection', 'Mask', 'Masked']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=9, fontweight='bold')
for row_i, (c, pos) in enumerate(cam_select):
    img  = attacked_imgs[pos]
    ce   = align_cam(cd2_full['cam_en_en'][pos])
    cz   = align_cam(cd2_full['cam_zh_zh'][pos])
    inter = n_cam_intersection(cd2_full['cam_en_en'][pos], cd2_full['cam_zh_zh'][pos])
    mask = masks_2mod[pos]
    panels = [img, overlay_cam(img, ce), overlay_cam(img, cz),
              overlay_cam(img, inter), mask_overlay(img, mask), masked_2mod[pos]]
    for col_i, panel in enumerate(panels):
        ax = axes[row_i, col_i]
        ax.imshow(panel if not isinstance(panel, np.ndarray) else panel)
        ax.axis('off')
    axes[row_i, 0].set_ylabel(CLASSES['en'][c], fontsize=8, rotation=0, labelpad=36, va='center')
plt.suptitle('2-mod CAM defence examples (multilingual attack)', fontsize=11, y=1.002)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR_2MOD}/mask_examples.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Saved -> {RESULTS_DIR_2MOD}/mask_examples.png')

Saved -> results/cam_2mod/accuracy_delta_matrix.png
Saved -> results/cam_4mod/accuracy_delta_matrix.png
Saved -> results/cam_2mod/mask_examples.png


## 11. Summary

In [19]:
print('\n=== Multilingual CAM Defence Summary ===')
for label, res in [('2-mod', res_2mod), ('4-mod', res_4mod)]:
    print(f'\n{label}:')
    for ml in LANGS:
        d = res['defense'][ml]
        print(f'  model_{ml}: {100*d["baseline_acc"]:.1f}% -> {100*d["acc"]:.1f}%  '
              f'ASR {100*d["baseline_asr"]:.1f}% -> {100*d["asr"]:.1f}%  '
              f'recovery={100*d["recovery_rate"]:.1f}%')


=== Multilingual CAM Defence Summary ===

2-mod:
  model_en: 4.3% -> 32.0%  ASR 95.5% -> 34.0%  recovery=30.4%
  model_zh: 7.3% -> 34.3%  ASR 92.7% -> 44.4%  recovery=32.3%

4-mod:
  model_en: 4.3% -> 29.8%  ASR 95.5% -> 40.6%  recovery=27.9%
  model_zh: 7.3% -> 31.9%  ASR 92.7% -> 50.9%  recovery=29.1%
